# exp4 — YOLOv11n

Same configuration as `exp2` (dataset version 2, 100 epochs, imgsz 640, batch 16, seed 42) — the only variable changed is the model architecture (`yolov8n.pt` → `yolo11n.pt`). Do not retrain exp2, which you already have ready.

In [10]:
!pip install -q ultralytics roboflow torch

In [11]:
from roboflow import Roboflow
from google.colab import userdata

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)  # use Colab Secrets / environment variable, never hardcoded in the saved notebook
project = rf.workspace("pedros-workspace-1w4rz").project("my-first-project-kfo4l")

# SAME version used in exp2 (original dataset, not the augmented fork
# used in exp3) — this ensures that the only variable changing
# between exp2 and exp4 is the model architecture.
version = project.version(2)
dataset = version.download("yolov8")

print(project.versions())

loading Roboflow workspace...
loading Roboflow project...
[<roboflow.core.version.Version object at 0x7ca2fc848510>, <roboflow.core.version.Version object at 0x7ca2fcd0b1d0>]


In [12]:
import torch

def select_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = select_device()
device

device(type='cuda')

In [13]:
# ---- Data integrity guard ----
# The project has already had a real incident of exp2/exp3 mixup (Section 7.1 of the
# report). This assert confirms that the downloaded dataset is indeed
# version 2 (the same as exp2) before spending 100 GPU epochs
# training with the wrong dataset.
expected_version_marker = "-2"  # Roboflow includes the version in the exported folder name
assert expected_version_marker in dataset.location, (
    f"INTEGRITY ERROR: expected dataset version 2 (same as exp2), "
    f"but the downloaded path was '{dataset.location}'. Stop and check before training."
)
print(f"✅ Dataset confirmed: {dataset.location}")

✅ Dataset confirmed: /content/My-First-Project-2


In [14]:
from ultralytics import YOLO

# The only real change compared to exp2: the model architecture.
model = YOLO('yolo11n.pt')

results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,          # same as exp2
    imgsz=640,           # same as exp2
    batch=16,            # same as exp2
    device=device,
    seed=42,             # same as exp2 — reproducibility
    project='runs', name='exp4',
)

print("NB COMPLETED SUCCESSFULLY — READY FOR field validation on exp4")

Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/My-First-Project-2/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp4, nbs=64, nms=None, opset=None, op

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
import os
import shutil

results_path = 'runs/detect/runs/exp4'

# Let's check if the directory 'runs/detect/runs/exp4' exists
if os.path.exists(results_path):
    print(f"Directory '{results_path}' found. Proceeding to zip.")
    shutil.make_archive('exp4_zipped', 'zip', results_path)
    print(f'Folder "{results_path}" successfully zipped to "exp4_zipped.zip"')
elif os.path.exists('runs/detect'):
    print("Error: Directory 'runs/detect/runs/exp4' not found. It seems 'exp4' is not directly under 'runs/detect'.")
    print("Contents of 'runs/detect' directory:")
    print(os.listdir('runs/detect'))
else:
    print(f"Error: Directory '{results_path}' not found. Please ensure 'train_exp4' cell executed successfully and created this directory.")
    print("Current working directory files:")
    print(os.listdir('.'))


Directory 'runs/detect/runs/exp4' found. Proceeding to zip.
Folder "runs/detect/runs/exp4" successfully zipped to "exp4_zipped.zip"


In [23]:
from google.colab import files
files.download('exp4_zipped.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
import os
print(os.listdir('.'))

['.config', 'runs', 'weights', 'exp4_zipped.zip', 'My-First-Project-2', 'drive', 'yolo11n.pt', 'sample_data']
